# Data Preparation: Standardizing Column Names
Here we load the raw CSV files, standardize their column names (lowercase, remove special characters, replace spaces with underscores), and save the cleaned versions to `data/02_processed/`.

In [1]:
import re
import pandas as pd
import numpy as np
from pathlib import Path

raw_dir = Path('../data/01_raw')
processed_dir = Path('../data/02_processed')
processed_dir.mkdir(parents=True, exist_ok=True)

def standardize_columns(df):
    """Cleans column strings to make them easy to work with in pandas"""
    new_cols = []
    for col in df.columns:
        cleaned = col.lower().strip()
        cleaned = re.sub(r'[^a-z0-9]+', '_', cleaned)
        cleaned = re.sub(r'_+', '_', cleaned)
        cleaned = cleaned.strip('_')
        new_cols.append(cleaned)
    df.columns = new_cols
    return df

def clean_values(df):
    """Standardizes cell values to fix specific categorical fuck-ups."""
    # List of terms that are essentially NULLs
    pseudo_nulls = ['not mentioned', 'not discussed', 'not applicable', 'n/a', 'none', 'nan']
    
    string_cols = df.select_dtypes(include=['object', 'string']).columns
    
    for col in string_cols:
        # 1. Lowercase and strip everything to prevent case-mismatches
        df[col] = df[col].astype(str).str.lower().str.strip()
        
        # 2. Treat 'nan' strings (which pandas can create when casting to str) as actual NaNs
        df.loc[df[col] == 'nan', col] = np.nan
        
        # 3. Standardize Pseudo-Nulls -> Actual NaNs
        df.loc[df[col].isin(pseudo_nulls), col] = np.nan
        
        # 4. Standardize Yes/No fields that got NLP text attached to them (e.g. "Yes, they asked for a discount")
        # If the text starts with 'yes' or 'no', convert to a strict '1' or '0' (string for now)
        if df[col].notna().any():
            is_yes = df[col].str.startswith('yes', na=False)
            is_no = df[col].str.startswith('no', na=False)
            
            # Apply strict mapping if it exhibits Yes/No behavior
            if is_yes.any() or is_no.any():
                df.loc[is_yes, col] = '1'
                df.loc[is_no, col] = '0'
        
        # 5. Fix Call Direction manually
        if 'direction' in col:
            df[col] = df[col].replace({'out_bound': 'outbound', 'in_bound': 'inbound'})
            
    return df

def clean_data_structure(df):
    """Drops completely empty columns and handles paragraph/text columns."""
    # 1. Drop columns where EVERYTHING (or >99%) is NULL
    missing_pct = df.isnull().mean()
    cols_to_drop = missing_pct[missing_pct > 0.99].index.tolist()
    if cols_to_drop:
        print(f"  -> Dropping {len(cols_to_drop)} mostly-NULL columns.")
        df = df.drop(columns=cols_to_drop)
        
    # 2. Handle Text Paragraphs
    object_cols = df.select_dtypes(include=['object', 'string']).columns
    for col in object_cols:
        # Ignore ID and date columns from being flagged as paragraphs
        if any(x in col for x in ['id', 'ref', 'date', 'time', 'month', 'year']):
            continue
            
        sample = df[col].dropna()
        if not sample.empty:
            # If the average word count is high, or string length is consistently long, it's free text.
            max_len = sample.astype(str).str.len().max()
            unique_count = sample.nunique()
            
            # If a column has over 50 unique long text strings, it's definitely a bot note
            if max_len > 75 and unique_count > 50: 
                print(f"  -> Converting paragraph column '{col}' to a boolean flag.")
                df[f"has_{col}_notes"] = df[col].notna().astype(int)
                df = df.drop(columns=[col])
                
    return df



In [2]:
files = ['raw_billings.csv', 'raw_renewal_calls.csv', 'raw_cc_calls.csv', 'raw_emails.csv']

for file in files:
    print(f"Processing {file}...")
    try:
        # Load it cleanly without memory issues
        if "csv" in file:
            df = pd.read_csv(raw_dir / file, low_memory=False)
        else:
            df = pd.read_excel(raw_dir / file)
    except FileNotFoundError:
        print(f"Skipping {file} - File not found.\n")
        continue
    
    # 1. Clean Column Names
    df = standardize_columns(df)
    
    # 2. Drop 'unnamed' columns that snuck in
    df = df.loc[:, ~df.columns.str.startswith('unnamed')]
    
    # 3. Clean and Unify the Values (the "Fuck Ups")
    print("  -> Unifying values (fixing Out_Bound, converting Yes/No etc.)")
    df = clean_values(df)
    
    # 4. Clean Structure (Dropping 99% NULLs, flag massive bot paragraphs)
    print("  -> Cleaning Structure...")
    df = clean_data_structure(df)
    
    # Save standardized file to processed folder
    save_path = processed_dir / "processed_" / file
    df.to_csv(save_path, index=False)
    print(f"Saved to {save_path} with {len(df.columns)} columns.\n")
    print("-" * 40)

Processing raw_billings.csv...
Skipping raw_billings.csv - File not found.

Processing raw_renewal_calls.csv...
Skipping raw_renewal_calls.csv - File not found.

Processing raw_cc_calls.csv...
Skipping raw_cc_calls.csv - File not found.

Processing raw_emails.csv...
Skipping raw_emails.csv - File not found.

